# ДОЭ: TIE4 → Physics-Guided CNN + Fourier-MLP + TT → ASM

**Одна модель `full` — CNN + Fourier + Tensor Train.** Это тот же вариант, который использовался в последней абляции, без замены архитектуры. 

- **100 обновлений весов на каждый комплект** из четырёх изображений (`z1`–`z4`), каждый комплект обучается независимо.
- На каждом обновлении — **три последовательных уточнения фазы** с физическим градиентом ASM и общими весами сети.
- **Fourier:** 64 моды; **Tensor Train:** ранг 6; **CNN:** 32 канала.
- Используется четырёхточечная схема TIE4 и обратное распространение до входного экрана.
- Исходные интенсивности изображений **не нормируются**. ASM не подгоняется по яркости и не возмещает уход света за рабочую область.
- Сохраняются лучшие 8-битные ДОЭ, веса, история метрик, время и численность вызовов ASM.
- `smoke-test` и сокращения режима обучения **нет**. Для запуска на твоих данных нужен корректный путь `FOLDER_PATH`.

**Важно:** 8-битные PNG рассматриваются как *линейные* карты интенсивностей. `Pearson`, `SSIM` и `RMSE` вычисляются без индивидуальной подгонки масштаба изображений.


In [ ]:
from __future__ import annotations
import math, os, time, csv, copy, warnings, random
from pathlib import Path
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.sparse.linalg import cg, LinearOperator
from scipy.ndimage import gaussian_filter
import torch
from torch import nn
from torch.nn import functional as F

# ----- ПУТЬ: оставлен точно как в test(1).py -----
FOLDER_PATH = Path(r"D:/Desktopd/Хахатон_оптика_ДОЭ/doe_z1/photos")
OUT = Path('./DOE_CNN_Fourier_TT_100steps_results')
OUT.mkdir(parents=True, exist_ok=True)
OUTPUT_FOLDER = OUT   # требуется оригинальному discover_dataset()

SEED = 20261108
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N = 128
DX = 10e-6
WAVELENGTH = 532e-9
Z_LIST = (0.0241, 0.0253, 0.0265, 0.0277)
Z_PLANES = Z_LIST   # требуется оригинальному discover_dataset()
PAD_FACTOR = 4
INCIDENT_AMPLITUDE = 1.0
SSIM_DATA_RANGE = 255.0
TT_RANK = 6
FOURIER_MODES = 64
INNER_STEPS = 3
STEPS_PER_CASE = 100
EVAL_EVERY = 10
LEARNING_RATE = 3e-4
MAX_PHASE_STEP_RAD = 0.20
BASE_GRADIENT_STEP_RAD = 0.12
BACKTRACK_FACTORS = (1.0, 0.5, 0.25, 0.125, 0.0625, 0.0)
QAT_LAST_STEPS = 10
GRAD_CLIP = 1.0
RUN_VARIANTS = ('full',)
TIE_SMOOTH_PX = 1.0
TIE_REG = 4e-3
TIE_FLOOR_FRACTION = 1e-3
TIE_PHASE_STRENGTH = 1.0

assert len(Z_LIST)==4 and np.allclose(np.diff(Z_LIST),Z_LIST[1]-Z_LIST[0],atol=1e-12,rtol=1e-8)
assert N>1 and PAD_FACTOR>=2 and INCIDENT_AMPLITUDE>0
assert STEPS_PER_CASE <= 100, 'Бюджет: не более 100 обновлений на комплект'
assert set(RUN_VARIANTS).issubset({'cnn','cnn_fourier','full'})
h = Z_LIST[1]-Z_LIST[0]
Z_C = 0.5*(Z_LIST[1]+Z_LIST[2])
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if DEVICE.type=='cuda': torch.cuda.manual_seed_all(SEED)
print('Device:',DEVICE,'| N:',N,'| planes:',Z_LIST,'| padding:',PAD_FACTOR)
print('Folder:',FOLDER_PATH,'exists:',FOLDER_PATH.is_dir(),'cwd:',Path.cwd())
print('Variants:',RUN_VARIANTS,'| steps per variant:',STEPS_PER_CASE, '| TOTAL updates per case:',len(RUN_VARIANTS)*STEPS_PER_CASE, '| inner steps:',INNER_STEPS)


## 1. Физически согласованный ASM: padding, выход света из рабочей апертуры, спектральное ограничение и эванесцентные моды

In [ ]:
class ASM(nn.Module):
    def __init__(self, n, dx, wavelength, z_list, pad_factor=4, bandlimit=True):
        super().__init__()
        assert n>0 and dx>0 and wavelength>0 and pad_factor>=2
        self.n=int(n); self.p=int(pad_factor*n); self.a=(self.p-self.n)//2
        self.dx=float(dx); self.wavelength=float(wavelength)
        self.z_list=tuple(float(z) for z in z_list)
        self.bandlimit=bool(bandlimit)
        self.asm_calls=0  # диагностический счётчик, на физику не влияет
        f=torch.fft.fftfreq(self.p,d=dx,dtype=torch.float64)
        FY,FX=torch.meshgrid(f,f,indexing='ij')
        k=2*math.pi/wavelength
        q2=k*k-(2*math.pi*FX)**2-(2*math.pi*FY)**2
        kz=torch.sqrt(q2.clamp_min(0.))
        alpha=torch.sqrt((-q2).clamp_min(0.))
        propagating=(q2>=0)
        H=[]
        for z in self.z_list:
            if z==0: transfer=torch.ones_like(kz,dtype=torch.complex128)
            else:
                transfer=torch.exp(1j*z*(kz-k))
                if z>0:
                    transfer=transfer*torch.exp(-z*alpha)
                else:
                    transfer=transfer*propagating.to(transfer.dtype)  # устойчивый обратный оператор
                if bandlimit:
                    L=self.p*dx
                    fcut=1/(wavelength*math.sqrt(1+(2*abs(z)/L)**2))
                    transfer=transfer*((FX.abs()<=fcut)&(FY.abs()<=fcut))
            H.append(transfer.to(torch.complex64))
        self.register_buffer('transfer',torch.stack(H),persistent=False)

    def forward_full(self,field):
        self.asm_calls += 1
        assert field.shape==(self.n,self.n)
        left=self.a; right=self.p-self.n-left
        padded=F.pad(field,(left,right,left,right))
        spec=torch.fft.fft2(padded,norm='ortho')
        return torch.fft.ifft2(spec[None]*self.transfer,norm='ortho')

    def forward(self,field):
        full=self.forward_full(field)
        a=self.a
        return full[:,a:a+self.n,a:a+self.n]

    def intensities(self,phase,quantized=False):
        if quantized: phase=quantize_8bit_ste(phase)
        field=INCIDENT_AMPLITUDE*torch.exp(1j*phase)  # фиксированная входная амплитуда; без перенормировки
        return self(field).abs().square()

def quantize_8bit_ste(phase):
    step=2*math.pi/256
    v=torch.remainder(phase,2*math.pi)
    rounded=torch.round(v/step)*step
    return v+(rounded-v).detach()  # точная дискретизация в forward; STE в backward

prop=ASM(N,DX,WAVELENGTH,Z_LIST,PAD_FACTOR).to(DEVICE)
inverse_prop=ASM(N,DX,WAVELENGTH,(-Z_C,),PAD_FACTOR).to(DEVICE)


## 2. Загрузчик из `test(1).py`: имена `z1…z4` во вложенных папках, исходная шкала пикселей

In [ ]:
# Загрузчик из test(1).py: имена, расширения, поиск папок без изменений.
# Единственное отличие по запросу: read_intensity возвращает СЫРЫЕ значения
# вместо нормировки array / max(array).

IMAGE_EXTENSIONS = {
    ".png", ".jpg", ".jpeg",
    ".bmp", ".tif", ".tiff", ".npy"
}

# Previous baseline on the original 10 examples.

# ============================================================
# 2. DATASET
# ============================================================

def read_intensity(path):
    if path.suffix.lower() == ".npy":
        array = np.load(path)
    else:
        with Image.open(path) as image:
            if image.mode in (
                "RGB", "RGBA", "P", "LA"
            ):
                image = image.convert("L")

            array = np.asarray(
                image,
                dtype=np.float32
            )

    array = np.asarray(
        array,
        dtype=np.float32
    )

    if array.ndim != 2:
        raise ValueError(
            f"Expected grayscale 2D image: {path}"
        )

    if array.shape != (N, N):
        raise ValueError(
            f"Expected {N}x{N}: {path}, "
            f"got {array.shape}"
        )

    if not np.isfinite(array).all():
        raise ValueError(
            f"Non-finite values: {path}"
        )

    array = np.maximum(array, 0.0)

    maximum = float(array.max())

    if maximum <= 0:
        raise ValueError(
            f"Completely black image: {path}"
        )

    return array  # сырые интенсивности; без деления на максимум


def discover_dataset(root):
    root = Path(root)

    if not root.is_dir():
        raise FileNotFoundError(
            f"Папка датасета не найдена: {root}"
        )

    names = []
    targets = []

    output_path = OUTPUT_FOLDER.resolve()

    directories = sorted(
        (p for p in root.rglob("*") if p.is_dir()),
        key=lambda p: str(p).lower()
    )

    for directory in directories:
        resolved = directory.resolve()

        if (
            resolved == output_path
            or output_path in resolved.parents
        ):
            continue

        plane_files = {
            f"z{i}": []
            for i in range(1, 5)
        }

        for path in directory.iterdir():
            if not path.is_file():
                continue

            if path.suffix.lower() not in IMAGE_EXTENSIONS:
                continue

            stem = path.stem.lower()

            if stem in plane_files:
                plane_files[stem].append(path)

        if not any(plane_files.values()):
            continue

        files = []

        for i in range(1, 5):
            key = f"z{i}"
            matches = plane_files[key]

            if len(matches) != 1:
                raise ValueError(
                    f"Папка: {directory}\n"
                    f"Для {key} найдено "
                    f"{len(matches)} файлов."
                )

            files.append(matches[0])

        sample = np.stack(
            [read_intensity(p) for p in files],
            axis=0
        )

        targets.append(
            torch.from_numpy(sample)
        )

        names.append(
            str(directory.relative_to(root))
        )

    if not targets:
        raise RuntimeError(
            "Не найдены папки с z1, z2, z3, z4."
        )

    data = torch.stack(
        targets, dim=0
    ).float()

    print(f"Примеров: {len(names)}")
    print(f"Размер данных: {tuple(data.shape)}")

    for i, z in enumerate(Z_PLANES, start=1):
        print(f"z{i}: {z * 1000:.1f} mm")

    return names, data


# Вызываем исключительно исходный discover_dataset().
names, raw_targets = discover_dataset(FOLDER_PATH)
samples = [{'name':name, 'target':raw.to(DEVICE,dtype=torch.float32),
            'tie_input':raw.numpy().astype(np.float64)}
           for name,raw in zip(names,raw_targets)]
print('Cases:',len(samples),'| no intensity normalization')
for sample in samples:
    print(sample['name'], 'min/max:',float(sample['target'].min()),float(sample['target'].max()))


## 3. TIE4-инициализация (все 4 плоскости)

Для равномерного шага $h$ берём среднюю плоскость $z_c=(z_2+z_3)/2$:
$$\partial_z I(z_c)\approx(I_1-27I_2+27I_3-I_4)/(24h),$$
$$I(z_c)\approx(-I_1+9I_2+9I_3-I_4)/16.$$
Решаем $-\nabla\cdot(I_c\nabla\phi_c)=k\partial_z I$ методом CG с регуляризацией и нулевым потоком через боковую границу TIE. Затем обратно распространяем комплексное поле ASM к экрану и извлекаем фазу. Порог `Ic` применяется **только внутри TIE-решателя**, исходные изображения не меняются.

In [ ]:
def derivative_4point(images,h):
    assert images.shape[0]==4 and h>0
    return (images[0]-27*images[1]+27*images[2]-images[3])/(24*h)

def midpoint_4point(images):
    assert images.shape[0]==4
    return (-images[0]+9*images[1]+9*images[2]-images[3])/16

def build_tie_matrix(I,dx,reg_fraction=4e-3):
    """A = -div(I grad) + λ; natural zero-flux edges; row-major flatten."""
    n,m=I.shape
    assert n==m
    I=np.maximum(I,1e-4).astype(np.float64)
    # Гармоническое среднее на гранях стабильнее при резком контрасте.
    wx=2*I[:,:-1]*I[:,1:]/(I[:,:-1]+I[:,1:]+1e-12)/dx**2
    wy=2*I[:-1,:]*I[1:,:]/(I[:-1,:]+I[1:,:]+1e-12)/dx**2
    pos=np.arange(n*n).reshape(n,n)
    diag=np.full(n*n,reg_fraction*np.mean(I)/dx**2,dtype=np.float64)
    rows=[];cols=[];vals=[]
    for a,b,w in ((pos[:,:-1],pos[:,1:],wx),(pos[:-1,:],pos[1:,:],wy)):
        ia=a.ravel(); ib=b.ravel(); wij=w.ravel()
        diag[ia]+=wij; diag[ib]+=wij
        rows.extend([ia,ib]); cols.extend([ib,ia]); vals.extend([-wij,-wij])
    rows.append(np.arange(n*n));cols.append(np.arange(n*n));vals.append(diag)
    rows=np.concatenate(rows);cols=np.concatenate(cols);vals=np.concatenate(vals)
    A=sparse.coo_matrix((vals,(rows,cols)),shape=(n*n,n*n)).tocsr()
    return A

def solve_tie_phase(Ic,dIdz,dx,wavelength,reg=4e-3):
    k=2*math.pi/wavelength
    A=build_tie_matrix(Ic,dx,reg)
    rhs=k*(dIdz-dIdz.mean()).ravel()  # совместимость с Neumann
    diagonal=A.diagonal()
    pre=LinearOperator(A.shape,matvec=lambda x:x/diagonal)
    phi,info=cg(A,rhs,M=pre,rtol=2e-5,atol=0,maxiter=1500)
    if info!=0: warnings.warn(f'TIE CG did not fully converge: info={info}')
    phi=phi.reshape(Ic.shape)
    phi-=phi.mean() # global phase gauge
    residual=float(np.linalg.norm(A@phi.ravel()-rhs)/(np.linalg.norm(rhs)+1e-20))
    return phi,residual


def tie_initialization_single(tie_images, index=0):
    im=np.stack([gaussian_filter(x,TIE_SMOOTH_PX,mode='constant',cval=0) for x in tie_images])
    dIdz=derivative_4point(im,h)
    Ic_raw=midpoint_4point(im)
    negative_fraction=float((Ic_raw<=0).mean())
    # Только для устойчивости PDE заменяем нулевую/отрицательную Ic малым
    # порогом, общим внутри данного TIE-решения. Исходные цели не меняем.
    floor=max(1e-9,TIE_FLOOR_FRACTION*max(float(Ic_raw.max()),0.))
    Ic=np.maximum(Ic_raw,floor)
    phi_c,residual=solve_tie_phase(Ic,dIdz,DX,WAVELENGTH,TIE_REG)
    with torch.no_grad():
        uc=torch.tensor(np.sqrt(Ic)*np.exp(1j*phi_c),device=DEVICE,dtype=torch.complex64)
        u0=inverse_prop(uc)[0]
        phase=TIE_PHASE_STRENGTH*torch.angle(u0).detach().float()
    if not torch.isfinite(phase).all():raise FloatingPointError('Nonfinite TIE phase')
    return phase,{'TIE_PDE_residual':residual,'TIE_negative_Ic_fraction':negative_fraction,
                  'phase_range':float(phi_c.max()-phi_c.min())},(dIdz,Ic,phi_c)


prop = ASM(N,DX,WAVELENGTH,Z_LIST,PAD_FACTOR).to(DEVICE)
inverse_prop = ASM(N,DX,WAVELENGTH,(-Z_C,),PAD_FACTOR).to(DEVICE)

def tie_initialization_single(tie_images, index=0):
    im=np.stack([gaussian_filter(x,TIE_SMOOTH_PX,mode='constant',cval=0) for x in tie_images])
    dIdz=derivative_4point(im,h)
    Ic_raw=midpoint_4point(im)
    negative_fraction=float((Ic_raw<=0).mean())
    floor=max(1e-9,TIE_FLOOR_FRACTION*max(float(Ic_raw.max()),0.0))
    Ic=np.maximum(Ic_raw,floor)
    phi_c,resid=solve_tie_phase(Ic,dIdz,DX,WAVELENGTH,TIE_REG)
    with torch.no_grad():
        uc=torch.tensor(np.sqrt(Ic)*np.exp(1j*phi_c),device=DEVICE,dtype=torch.complex64)
        u0=inverse_prop(uc)[0]
        phase=TIE_PHASE_STRENGTH*torch.angle(u0).float()
    if not torch.isfinite(phase).all(): raise FloatingPointError('TIE yielded NaN/Inf')
    return phase, {'tie_residual':resid,'negative_center_fraction':negative_fraction}

for i,s in enumerate(samples):
    s['tie_phase'],s['tie_info']=tie_initialization_single(s['tie_input'],i)
    print(f'TIE {i+1}/{len(samples)}: {s["name"]}: {s["tie_info"]}')


## 4. Pearson, SSIM, RMSE, NRMSE — без переподгонки выходов ASM или исходных PNG

In [ ]:
def metrics_and_loss(intensities,target):
    assert intensities.shape==target.shape==(4,N,N)
    X=intensities.flatten(1); Y=target.flatten(1)
    xc=X-X.mean(dim=1,keepdim=True); yc=Y-Y.mean(dim=1,keepdim=True)
    pearson=(xc*yc).sum(dim=1)/(xc.norm(dim=1)*yc.norm(dim=1)+1e-8)
    rmse=(intensities-target).square().mean(dim=(-1,-2)).sqrt()
    nrmse=rmse/(target.square().mean(dim=(-1,-2)).sqrt()+1e-8)
    kernel_coord=torch.arange(-3,4,device=target.device,dtype=target.dtype)
    gaussian=torch.exp(-kernel_coord.square()/(2*1.5**2)); gaussian=gaussian/gaussian.sum()
    win=(gaussian[:,None]*gaussian[None,:])[None,None]
    def blur(x): return F.conv2d(F.pad(x,(3,3,3,3),mode='reflect'),win)
    a=intensities[:,None]; b=target[:,None]
    ma=blur(a);mb=blur(b)
    va=(blur(a*a)-ma*ma).clamp_min(0.0)
    vb=(blur(b*b)-mb*mb).clamp_min(0.0)
    cov=blur(a*b)-ma*mb
    c1=(0.01*SSIM_DATA_RANGE)**2; c2=(0.03*SSIM_DATA_RANGE)**2
    ssim=((2*ma*mb+c1)*(2*cov+c2))/((ma*ma+mb*mb+c1)*(va+vb+c2)+1e-12)
    ssim=ssim.mean(dim=(1,2,3))
    per_plane=0.65*(1-pearson)+0.30*(1-ssim)+0.05*torch.log1p(nrmse)
    loss=per_plane.mean()
    return loss,{'P':pearson,'SSIM':ssim,'RMSE':rmse,'NRMSE':nrmse,
                 'Q':0.5*(pearson+ssim),'loss_per_plane':per_plane}

@torch.no_grad()
def evaluate_phase(phi,target,quantized=False):
    I=prop.intensities(phi,quantized=quantized)
    loss,met=metrics_and_loss(I,target)
    return dict(loss=float(loss),P=float(met['P'].mean()),SSIM=float(met['SSIM'].mean()),
                RMSE=float(met['RMSE'].mean()),NRMSE=float(met['NRMSE'].mean()),
                Qmin=float(met['Q'].min()),Pmin=float(met['P'].min()),
                Smin=float(met['SSIM'].min()))

print('TIE baselines:')
for s in samples:
    s['tie_metrics']=evaluate_phase(s['tie_phase'],s['target'])
    print(s['name'],s['tie_metrics'])


## 5. Физический градиент и Fourier-проекция

На каждой внутренней итерации получаем $g=\partial\mathcal L/\partial\phi$ **непосредственно через ASM**. Градиент вычисляется на отделённой копии текущей фазы, так что при обучении нет вторых производных. Сеть видит пространственное поле градиента и проекции $\langle g,f_k\rangle$ на Fourier-моды. Масштабируется только **входной градиент**, не интенсивности.

In [ ]:
def make_fourier_basis(n,modes,device):
    grid=torch.arange(n,device=device,dtype=torch.float32)/n
    yy,xx=torch.meshgrid(grid,grid,indexing='ij')
    pairs=[]; bound=max(5,int(math.ceil(math.sqrt(modes))))
    for kx in range(-bound,bound+1):
        for ky in range(-bound,bound+1):
            if kx>0 or (kx==0 and ky>0): pairs.append((kx,ky))
    pairs.sort(key=lambda a:(a[0]**2+a[1]**2,a[0],a[1]))
    functions=[]; names=[]
    for kx,ky in pairs:
        omega=2*math.pi*(kx*xx+ky*yy)
        functions.extend([math.sqrt(2)*torch.cos(omega),math.sqrt(2)*torch.sin(omega)])
        names.extend([(kx,ky,'cos'),(kx,ky,'sin')])
        if len(functions)>=modes: break
    basis=torch.stack(functions[:modes],dim=0)
    G=torch.einsum('khw,lhw->kl',basis,basis)/(n*n)
    assert torch.allclose(G,torch.eye(modes,device=device),atol=1e-4), 'Fourier basis not orthogonal'
    return basis,names[:modes]

FOURIER_BASIS,FOURIER_NAMES=make_fourier_basis(N,FOURIER_MODES,DEVICE)

# Только текущая фаза отсоединяется при вычислении физического градиента:
# никакого second-order autodiff через torch.autograd.grad.
def phase_physics_gradient(phase,target,quantized=False):
    with torch.enable_grad():
        probe=phase.detach().clone().requires_grad_(True)
        loss,_=metrics_and_loss(prop.intensities(probe,quantized=quantized),target)
        g=torch.autograd.grad(loss,probe,create_graph=False,retain_graph=False)[0]
    rms=g.square().mean().sqrt().clamp_min(1e-12)
    return (g/rms).detach()

XY=torch.linspace(-1,1,N,device=DEVICE,dtype=torch.float32)
YY,XX=torch.meshgrid(XY,XY,indexing='ij')
COORDS=torch.stack([XX,YY])[None]

def network_features(phase,target,pred,g):
    residual=target-pred
    return torch.cat([torch.cos(phase)[None,None],torch.sin(phase)[None,None],
                      target[None],pred[None],residual[None],g[None,None],COORDS],dim=1)


## 6. Архитектура CNN + Fourier-MLP + Tensor Train

Ровно **один** вариант `full`: свёрточный энкодер анализирует текущую фазу и физические признаки; Fourier-MLP получает Fourier-проекции текущей фазы и градиента (64 компоненты), TT синтезирует глобальную структурированную карту (ранг 6). Декодер формирует **одну** фазовую поправку. Дополнительной независимой фазовой коррекции из других стадий нет.


In [ ]:
class ConvBlock(nn.Module):
    def __init__(self,channels):
        super().__init__()
        self.net=nn.Sequential(nn.Conv2d(channels,channels,3,padding=1),
                               nn.GroupNorm(8,channels),nn.SiLU(),
                               nn.Conv2d(channels,channels,3,padding=1),
                               nn.GroupNorm(8,channels))
    def forward(self,x): return F.silu(x+self.net(x))

class PhysicalTensorFourier(nn.Module):
    """Same supernetwork initial weights; disabled branches feed literal zeros."""
    def __init__(self,n,rank,basis,variant='full',channels=32):
        super().__init__()
        assert variant in ('cnn','cnn_fourier','full')
        self.variant=variant; self.n=n; self.rank=rank
        self.use_fourier=variant in ('cnn_fourier','full')
        self.use_tt=variant=='full'
        self.encoder=nn.Sequential(nn.Conv2d(17,channels,3,padding=1),
                                   nn.GroupNorm(8,channels),nn.SiLU(),
                                   ConvBlock(channels),ConvBlock(channels))
        self.global_embed=nn.Sequential(nn.AdaptiveAvgPool2d((4,4)),nn.Flatten(),
                                        nn.Linear(channels*16,128),nn.SiLU())
        self.register_buffer('basis',basis.detach().clone(),persistent=False)
        K=basis.shape[0]
        self.fourier_mlp=nn.Sequential(nn.Linear(128+2*K,192),nn.SiLU(),
                                       nn.Linear(192,192),nn.SiLU(),nn.Linear(192,K))
        nn.init.zeros_(self.fourier_mlp[-1].weight)
        nn.init.zeros_(self.fourier_mlp[-1].bias)
        logn=round(math.log2(n)); assert n==2**logn, 'TT requires power-of-two image size'
        a=2**(logn//2); b=n//a
        self.core_shapes=[(1,a,rank),(rank,b,rank),(rank,a,rank),(rank,b,1)]
        self.tt_cores=nn.ParameterList()
        self.tt_heads=nn.ModuleList()
        for shape in self.core_shapes:
            self.tt_cores.append(nn.Parameter(0.12*torch.randn(shape)))
            head=nn.Linear(128,math.prod(shape))
            nn.init.zeros_(head.weight); nn.init.zeros_(head.bias)
            self.tt_heads.append(head)
        # IMPORTANT: exact same decoder input channels / shapes in ALL variants.
        self.decoder=nn.Sequential(nn.Conv2d(channels+3,channels,3,padding=1),
                                   nn.GroupNorm(8,channels),nn.SiLU(),
                                   ConvBlock(channels),nn.Conv2d(channels,1,3,padding=1))
        nn.init.zeros_(self.decoder[-1].weight)
        nn.init.zeros_(self.decoder[-1].bias)
        # The inactive blocks do not consume optimizer budget/weight decay.
        if not self.use_fourier:
            for p in self.fourier_mlp.parameters(): p.requires_grad_(False)
        if not self.use_tt:
            for p in self.tt_cores: p.requires_grad_(False)
            for p in self.tt_heads.parameters(): p.requires_grad_(False)
        if not (self.use_tt or self.use_fourier):
            for p in self.global_embed.parameters(): p.requires_grad_(False)

    def make_tt(self,code):
        cores=[]
        for base,head,shape in zip(self.tt_cores,self.tt_heads,self.core_shapes):
            cores.append(base+0.15*torch.tanh(head(code).reshape(shape)))
        A,B,C,D=cores
        return torch.einsum('ia,ajb,bkc,cl->ijkl',A[0],B,C,D[:,:,0]).reshape(self.n,self.n)

    def forward(self,features,g,phase):
        h=self.encoder(features)
        zero=torch.zeros_like(g)
        code=self.global_embed(h)[0] if (self.use_fourier or self.use_tt) else None
        if self.use_fourier:
            spec_g=torch.einsum('khw,hw->k',self.basis,g)/(self.n*self.n)
            spec_phi=torch.einsum('khw,hw->k',self.basis,phase)/(self.n*self.n)
            coefficients=0.2*torch.tanh(self.fourier_mlp(torch.cat([code,spec_g,spec_phi])))
            spectral=torch.einsum('k,khw->hw',coefficients,self.basis)/math.sqrt(len(coefficients))
        else:
            spectral=zero
        if self.use_tt:
            tensor_map=torch.tanh(self.make_tt(code))
        else:
            tensor_map=zero
        fused=self.decoder(torch.cat([h,g[None,None],spectral[None,None],tensor_map[None,None]],dim=1))[0,0]
        correction=MAX_PHASE_STEP_RAD*torch.tanh(fused)
        return correction-correction.mean()

def parameter_count(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
print('Model:',RUN_VARIANTS,'| Fourier modes:',FOURIER_MODES,'| TT rank:',TT_RANK)


## 7. Внутренний цикл с backtracking по настоящему ASM-loss

$\phi_{t+1}=\phi_t+\alpha_t[-\eta\,g_t^{\rm unit}+\Delta\phi_{\rm net}(\phi_t,g_t)]$.

$\alpha_t$ выбирается из фиксированного списка **по физической функции потерь**: если полный шаг неудачен, проверяются меньшие; нулевой шаг допустим. Это устраняет искусственные ухудшения **на внутренних шагах**. Во время обучения результат конечного шага переносится на следующую эпоху; нет отдельных фазовых стадий и нет повторного добавления ранее внесённой поправки.

In [ ]:
def choose_step(current_phase,delta,target,quantized):
    """No gradients through choice of scale; gradients through selected candidate are preserved."""
    with torch.no_grad():
        base_loss,_=metrics_and_loss(prop.intensities(current_phase.detach(),quantized=quantized),target)
        best_alpha=0.0; best_loss=float(base_loss)
        for alpha in BACKTRACK_FACTORS:
            if alpha==0.0: continue
            trial=current_phase.detach()+alpha*delta.detach()
            trial_loss,_=metrics_and_loss(prop.intensities(trial,quantized=quantized),target)
            score=float(trial_loss)
            if score<best_loss-1e-7:
                best_alpha=float(alpha);best_loss=score
                break  # первый приемлемый, а не глобальный минимум сетки
    return best_alpha


def iterate_phase(phase,target,model,steps=INNER_STEPS,quantized=False):
    """Several physical corrections with shared trainable weights; target unchanged."""
    losses=[]; alphas=[]; correction_ratios=[]
    for iteration in range(steps):
        # Physics gradient: phase detached ONLY inside the gradient probe.
        gradient=phase_physics_gradient(phase,target,quantized=quantized)
        with torch.no_grad():
            predicted=prop.intensities(phase.detach(),quantized=quantized)
        features=network_features(phase,target,predicted,gradient)
        learned_delta=model(features,gradient,phase)
        delta=-BASE_GRADIENT_STEP_RAD*gradient+learned_delta
        # Насколько реально нейросеть участвует в шаге относительно физического градиента?
        ratio=float((learned_delta.detach().square().mean().sqrt() /
                     ((BASE_GRADIENT_STEP_RAD*gradient).square().mean().sqrt()+1e-12)).item())
        correction_ratios.append(ratio)
        alpha=choose_step(phase,delta,target,quantized)
        phase=phase+alpha*delta
        loss,_=metrics_and_loss(prop.intensities(phase,quantized=quantized),target)
        losses.append(loss)
        alphas.append(alpha)
    weights=torch.arange(1,len(losses)+1,device=phase.device,dtype=phase.dtype)
    weights=weights/weights.sum()
    total=sum(w*loss for w,loss in zip(weights,losses))
    return total,phase,alphas,float(np.mean(correction_ratios))


## 8. Обучение — 100 шагов для одного комплекта

Каждый набор `z1`–`z4` имеет свою собственную сеть и оптимизатор. На каждом шаге считаем физический градиент по фазе, трижды уточняем фазу и выполняем backpropagation через принятые ASM-поправки. Промежуточная фаза сохраняется между шагами, а сеть не применяется повторно на границе стадий (таких стадий в этой архитектуре больше нет).


In [ ]:
def train_case(sample,variant,case_index):
    target=sample['target']
    tie_phase=sample['tie_phase'].detach()
    variant_seed=SEED + 10000*case_index  # SAME seed for all three variants of this case
    torch.manual_seed(variant_seed)
    model=PhysicalTensorFourier(N,TT_RANK,FOURIER_BASIS,variant).to(DEVICE)
    optimizer=torch.optim.AdamW((p for p in model.parameters() if p.requires_grad),lr=LEARNING_RATE,weight_decay=1e-5)
    case_dir=OUT/sample['name']/variant
    case_dir.mkdir(parents=True,exist_ok=True)
    current_phase=tie_phase.clone()
    best_phase=current_phase.clone()
    best_score=evaluate_phase(best_phase,target,quantized=True)
    best_model=copy.deepcopy({k:v.detach().cpu().clone() for k,v in model.state_dict().items()})
    history=[]
    start=time.perf_counter()
    calls_start=prop.asm_calls
    init=evaluate_phase(tie_phase,target,quantized=False)
    print(f'\n[{sample["name"]} / {variant}] parameters={parameter_count(model)}; initial P={init["P"]:.4f}, SSIM={init["SSIM"]:.4f}')
    for epoch in range(1,STEPS_PER_CASE+1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        qat=QAT_LAST_STEPS>0 and epoch>STEPS_PER_CASE-QAT_LAST_STEPS
        train_loss,proposed,accepted,net_to_grad_ratio=iterate_phase(current_phase.detach(),target,model,
                                                    steps=INNER_STEPS,quantized=qat)
        if not torch.isfinite(train_loss):
            raise FloatingPointError(f'Nonfinite loss at {sample["name"]} / {variant}, step {epoch}')
        # If no step was accepted, the network may receive zero gradient; this is valid.
        train_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP)
        optimizer.step()
        current_phase=proposed.detach()  # PHASE IS PERSISTENT BETWEEN EPOCHS
        if epoch==1 or epoch%EVAL_EVERY==0 or epoch==STEPS_PER_CASE:
            model.eval()
            metric=evaluate_phase(current_phase,target,quantized=False)
            metric_q=evaluate_phase(current_phase,target,quantized=True)
            row={'step':epoch,'time_s':time.perf_counter()-start,'variant':variant,
                 'train_loss':float(train_loss.detach()),'accepted_inner':sum(a>0 for a in accepted),
                 'step_scales':'/'.join(f'{a:g}' for a in accepted),
                 'asm_calls':prop.asm_calls-calls_start,
                 'net_to_grad_ratio':net_to_grad_ratio,
                 **metric, 'Qmin_8bit':metric_q['Qmin'],'P_8bit':metric_q['P'],
                 'SSIM_8bit':metric_q['SSIM'],'loss_8bit':metric_q['loss']}
            history.append(row)
            # Take best actually quantized solution; restore weights and phase consistently.
            if metric_q['loss']<best_score['loss']:
                best_score=metric_q
                best_phase=current_phase.clone()
                best_model=copy.deepcopy({k:v.detach().cpu().clone() for k,v in model.state_dict().items()})
            print(f'  {variant:>12s} {epoch:4d}/{STEPS_PER_CASE}: '
                  f'P={metric["P"]:.4f} S={metric["SSIM"]:.4f} '
                  f'Qmin={metric["Qmin"]:.4f} NRMSE={metric["NRMSE"]:.4f} '
                  f'accepted={sum(a>0 for a in accepted)}/{INNER_STEPS} '
                  f'net/gradient={net_to_grad_ratio:.3f} '
                  f't={row["time_s"]:.1f}s',flush=True)
    model.load_state_dict(best_model)
    with open(case_dir/'history.csv','w',newline='',encoding='utf-8-sig') as fp:
        writer=csv.DictWriter(fp,fieldnames=list(history[0]));writer.writeheader();writer.writerows(history)
    torch.save({'state_dict':best_model,'variant':variant,'case_name':sample['name'],
                'best_metrics':best_score,'config':{'N':N,'dx':DX,'wavelength':WAVELENGTH,
                'z_list':Z_LIST,'tt_rank':TT_RANK,'fourier_modes':FOURIER_MODES}},case_dir/'model.pt')
    np.save(case_dir/'phase_rad.npy',best_phase.detach().cpu().numpy())
    quantized=((best_phase.detach().cpu().remainder(2*math.pi)/(2*math.pi))*256).round().long()%256
    Image.fromarray(quantized.numpy().astype('uint8'),'L').save(case_dir/'doe_8bit.png')
    return {'case':sample['name'],'variant':variant,'steps':STEPS_PER_CASE,
            't_s':time.perf_counter()-start,'asm_calls':prop.asm_calls-calls_start,
                 'net_to_grad_ratio':net_to_grad_ratio,
            'trainable_parameters':parameter_count(model),
            'baseline_P':init['P'],
            'baseline_SSIM':init['SSIM'],**{f'best_{k}':v for k,v in best_score.items()}},history,best_phase.detach().clone()


## 9. Запуск на всех комплектах — только `full`, 100 шагов на комплект

Каждая вложенная папка с четырьмя `z1…z4` — отдельная оптическая задача. Работает загрузчик из `test(1).py` (оригинальная логика поиска, но без деления интенсивностей на максимум). Изменять путь нужно только в первой кодовой ячейке.


In [ ]:
all_results=[]; histories={}; phases={}
for i,s in enumerate(samples):
    for variant in ('full',):
        result,records,phase=train_case(s,variant,i)
        all_results.append(result);histories[(s['name'],variant)]=records
        phases[(s['name'],variant)]=phase
        with open(OUT/'summary.csv','w',newline='',encoding='utf-8-sig') as fp:
            writer=csv.DictWriter(fp,fieldnames=list(all_results[0]))
            writer.writeheader();writer.writerows(all_results)
print('Saved results:',OUT.resolve())


## 10. Графики и изображение четырёх плоскостей

На каждый комплект сохраняются `Pearson_vs_time.png`, `SSIM_vs_time.png`, `Qmin_vs_time.png`, `NRMSE_vs_time.png`, `four_planes.png`, `doe_8bit.png`, `phase_rad.npy`, `model.pt`, `history.csv`. В `four_planes.png` вторая строка показывает **фактическую** яркость ASM в той же шкале, что цель, а третья — отдельно автоконтрастную визуализацию *только для просмотра*, не для метрик.


In [ ]:
def plot_history(sample_name,variant,records):
    directory=OUT/sample_name/variant
    times=[r['time_s'] for r in records]
    for key,filename in [('P','Pearson_vs_time.png'),('SSIM','SSIM_vs_time.png'),
                         ('Qmin','Qmin_vs_time.png'),('NRMSE','NRMSE_vs_time.png')]:
        fig,ax=plt.subplots(figsize=(8,4.5))
        ax.plot(times,[r[key] for r in records],label=f'{variant}: {key}')
        ax.set(xlabel='Training time, seconds',ylabel=key,title=f'{sample_name}: {key}(t)')
        ax.grid(alpha=.3);ax.legend();fig.tight_layout();fig.savefig(directory/filename,dpi=160)
        plt.close(fig)

@torch.no_grad()
def plot_comparison(sample,variant,phase):
    d=OUT/sample['name']/variant
    pred=prop.intensities(phase,quantized=True).detach().cpu().numpy()
    ref=sample['target'].detach().cpu().numpy()
    fig,ax=plt.subplots(3,4,figsize=(15,9),squeeze=False)
    for j in range(4):
        ax[0,j].imshow(ref[j],cmap='magma',vmin=0,vmax=255)
        ax[0,j].set_title(f'Target z{j+1} [0,255]')
        ax[1,j].imshow(pred[j],cmap='magma',vmin=0,vmax=255)
        ax[1,j].set_title(f'ASM z{j+1} SAME scale')
        # Только визуальный autoscale для изучения формы, НЕ меняет метрики.
        ax[2,j].imshow(pred[j],cmap='magma')
        ax[2,j].set_title(f'ASM z{j+1} display autoscale')
        for r in range(3): ax[r,j].axis('off')
    fig.tight_layout();fig.savefig(d/'four_planes.png',dpi=145);plt.close(fig)

for s in samples:
    for variant in RUN_VARIANTS:
        plot_history(s['name'],variant,histories[(s['name'],variant)])
        plot_comparison(s,variant,phases[(s['name'],variant)])
print('All figures stored in:',OUT.resolve())


## Результаты и ограничения

- Напрямую оптимизируется одна фаза для четырёх плоскостей, а не отдельные фазы для каждой.
- Целевые массивы из PNG/NPY проходят в TIE/ASM/loss **без деления на максимум, среднее или сумму**. Локальная регуляризация TIE (floor в областях почти нулевой интенсивности) влияет только на стартовую фазу, но не изменяет исходные цели.
- `Qmin = min_j 0.5(P_j+SSIM_j)` — вспомогательная метрика худшей плоскости, `P`/`SSIM` в логах — средние по четырём плоскостям.
- Абсолютный RMSE при фиксированном входном поле может оставаться большим, даже когда Pearson высок; это не исправляется скрытым рескейлингом.
- Если нужно увеличить число шагов в будущем, достаточно изменить `STEPS_PER_CASE` (и соответствующий budget assert).
